# `Design Freeze`——把 `Stage 4/5` 的設計結果組裝成結構化 `JSON`

**這份 `JSON` 由計算過程直接產生,不是手動謄寫 `stage4_5` 算出的
數字**——重新執行同一套已驗證過的函式,組裝成單一的
`design_freeze.json`,供後續非線性分析(`Case-06.7`/`06.8`/`06.9`)
直接讀取,不再各自假設配筋。

**這份凍結涵蓋的範圍(誠實列出,不誇大)**:`PM` 互制、最小鋼筋比、
正負彎矩要求、`capacity-design shear`(梁`+`柱)、柱端圍束、梁柱
接頭剪力、`SCWB`(含頂層豁免條款)——共 `7` 類結構性檢核,`1F`/`2F`
兩層都覆蓋。**沒有涵蓋**:錨定/搭接長度、柱主筋沿全長排筋、接頭
橫向鋼筋排列細節——這些不在這次凍結範圍內,標注在 `assumptions`
欄位裡,不是隱藏起來。

In [1]:

import json, os
if not os.path.exists('rc_design.py'):
    !wget -q -O rc_design.py https://raw.githubusercontent.com/zhixiu0223/taiwan-seismic-code-calc/main/rc_design.py

from rc_design import design_column_PM, design_rebar

def Mn_at_Pu(curve, Pu_target):
    sc = sorted(curve, key=lambda p: p['Pn'])
    for i in range(len(sc)-1):
        if sc[i]['Pn'] <= Pu_target <= sc[i+1]['Pn']:
            t = (Pu_target-sc[i]['Pn'])/(sc[i+1]['Pn']-sc[i]['Pn'])
            return sc[i]['Mn'] + t*(sc[i+1]['Mn']-sc[i]['Mn'])
    return None

def compute_Mpr(As_cm2, b_cm, d_cm, fc=280.0, fy=4200.0):
    fy_pr = 1.25*fy
    a = As_cm2*fy_pr/(0.85*fc*b_cm)
    return (As_cm2*fy_pr*(d_cm - a/2)) * 9.80665e-5

def design_column_rebar(b_cm, h_cm, Pu_kN, Mu_kNm, fc=280.0, fy=4200.0, Es=2.0e6,
                          cover_to_center=4.0, rho_candidates=None):
    if rho_candidates is None:
        rho_candidates = [0.010, 0.012, 0.014, 0.016, 0.018, 0.020, 0.025, 0.030, 0.040]
    for rho in rho_candidates:
        As_bar = rho*b_cm*h_cm/8
        As_layers = [3*As_bar, 2*As_bar, 3*As_bar]
        r = design_column_PM(b_cm=b_cm, h_cm=h_cm, As_per_layer_cm2=As_layers,
                               cover_to_center_cm=cover_to_center, n_layers=3,
                               fc=fc, fy=fy, Es=Es, Pu_kN=Pu_kN, Mu_kNm=Mu_kNm)
        if r['within_envelope']:
            return dict(rho=rho, As_total=8*As_bar, r=r, Mn=Mn_at_Pu(r['curve'], Pu_kN))
    return None

def design_column_confinement(h_col_cm, Ln_cm, db_long_cm, fc=280.0, fyt=4200.0, cover_cm=4.0):
    lo = max(h_col_cm, Ln_cm/6, 45.0)
    bc = h_col_cm - 2*cover_cm
    so = min(max(10.0 + (35.0-bc)/3.0, 10.0), 15.0)
    s_max = min(h_col_cm/4, 6*db_long_cm, so)
    Ag = h_col_cm**2
    Ach = bc**2
    Ash_over_s_req = max(0.3*bc*(fc/fyt)*(Ag/Ach-1), 0.09*bc*fc/fyt)
    Ash_2leg = 2*0.7133
    s_final = min(Ash_2leg/Ash_over_s_req, s_max)
    return dict(lo=lo, s_max=s_max, s_final=s_final)

print("核心函式定義完成")


核心函式定義完成


## 幾何/材料/需求——延續 `Case-04.7`(`Stage 2/3`,四工具驗證過)的資料

In [2]:

GEOMETRY = dict(L_bay_m=6.0, h1_m=3.5, h2_m=3.5,
                 col_section_cm="40x40", beam_section_cm="30x50")
MATERIALS = dict(fc_kgf_cm2=280.0, fy_kgf_cm2=4200.0, E_kPa=2.463e7,
                  stiffness_reduction="beam 0.35Ig, column 0.7Ig (ACI 318 declared assumption, not unique true value)")

# governing demand: 來自Case-04.7四工具驗證過的canonical elastic model,
# D+L+E/D+L-E組合, 已連著load combination記錄
DEMANDS = {
    "1F_column": dict(Pu_kN=308.85, Mu_kNm=32.13, Vu_kN=12.94, governing_combo="D+L+E"),
    "2F_column": dict(Pu_kN=153.19, Mu_kNm=11.06, Vu_kN=7.95, governing_combo="D+L+E"),
    "1F_beam":   dict(Mu_neg_kNm=24.189, Mu_pos_kNm=24.168, Vu_kN=8.06,
                        note="D+L給出的樑端彎矩接近0(重力集中施加在柱頂節點, 非分佈載重), 正負彎矩幾乎完全由側推力決定"),
    "roof_beam": dict(Mu_neg_kNm=16.784, Mu_pos_kNm=16.784, Vu_kN=5.59),
}
print("需求資料組裝完成")


需求資料組裝完成


## 逐一重新執行設計計算(不是複製數字,是重新跑一次)

In [3]:

# --- 柱配筋 ---
result_1F_col = design_column_rebar(40.0, 40.0, DEMANDS["1F_column"]["Pu_kN"], DEMANDS["1F_column"]["Mu_kNm"])
result_2F_col = design_column_rebar(40.0, 40.0, DEMANDS["2F_column"]["Pu_kN"], DEMANDS["2F_column"]["Mu_kNm"])

# --- 梁配筋(正負彎矩分開設計) ---
r_1F_neg = design_rebar(DEMANDS["1F_beam"]["Mu_neg_kNm"], 30.0, 50.0, cover=4.0)
r_1F_pos = design_rebar(DEMANDS["1F_beam"]["Mu_pos_kNm"], 30.0, 50.0, cover=4.0)
r_roof_neg = design_rebar(DEMANDS["roof_beam"]["Mu_neg_kNm"], 30.0, 50.0, cover=4.0)
r_roof_pos = design_rebar(DEMANDS["roof_beam"]["Mu_pos_kNm"], 30.0, 50.0, cover=4.0)

print(f"1F柱: rho={result_1F_col['rho']:.3%}, Mn={result_1F_col['Mn']:.2f}kN-m")
print(f"2F柱: rho={result_2F_col['rho']:.3%}, Mn={result_2F_col['Mn']:.2f}kN-m")
print(f"1F樑上緣: As={r_1F_neg['As_provided']:.2f}cm2, 下緣: As={r_1F_pos['As_provided']:.2f}cm2")
print(f"屋頂樑上緣: As={r_roof_neg['As_provided']:.2f}cm2, 下緣: As={r_roof_pos['As_provided']:.2f}cm2")


1F柱: rho=1.000%, Mn=159.97kN-m
2F柱: rho=1.000%, Mn=136.83kN-m
1F樑上緣: As=5.73cm2, 下緣: As=5.73cm2
屋頂樑上緣: As=5.73cm2, 下緣: As=5.73cm2


In [4]:

# --- SCWB ---
# 定義統一成跟原始stage4_5 notebook完全一致: ratio = sum_Mnc/sum_Mnb
# (不把1.2併入分母), 判斷用ratio>=1.2——上一版曾經把1.2併入分母但
# 判斷閾值忘了同步改成1.0, 算出ratio=2.49(跟原始2.99不一致), 雖然
# 這次PASS結果剛好還對, 但邏輯本身是錯的, 已經修正並在這裡明確
# 記錄這個曾經發生過的不一致, 不是船過水無痕
sum_Mnc_1F_joint = result_1F_col['Mn'] + result_2F_col['Mn']
Mn_1F_beam = r_1F_neg['phiMn_provided']/0.9
scwb_1F_ratio = sum_Mnc_1F_joint/Mn_1F_beam  # 不含1.2, 跟原notebook定義一致

# 屋頂接頭: 檢查ACI 318-19 18.7.3.1豁免條款
limit_scwb_exempt = (40.0*40.0*280.0/10)*9.80665e-3
roof_scwb_exempt = DEMANDS["2F_column"]["Pu_kN"] < limit_scwb_exempt

print(f"1F接頭SCWB: sum(Mnc)={sum_Mnc_1F_joint:.2f}, ratio={scwb_1F_ratio:.2f}, PASS={scwb_1F_ratio>=1.2}")
print(f"屋頂接頭SCWB: 豁免(ACI 318-19 18.7.3.1, 柱不連續+Pu<{limit_scwb_exempt:.2f}kN)={roof_scwb_exempt}")

assert abs(scwb_1F_ratio - 2.99) < 0.1, "修正後應該重現原始notebook的2.99"
print(f"\n[驗證] 修正後ratio={scwb_1F_ratio:.2f}, 跟原notebook的2.99一致")


1F接頭SCWB: sum(Mnc)=296.80, ratio=2.99, PASS=True


屋頂接頭SCWB: 豁免(ACI 318-19 18.7.3.1, 柱不連續+Pu<439.34kN)=True

[驗證] 修正後ratio=2.99, 跟原notebook的2.99一致


In [5]:

# --- Capacity-design shear(梁) ---
Mpr_1F_beam = compute_Mpr(r_1F_neg['As_provided'], 30.0, r_1F_neg['d'])
Mpr_roof_beam = compute_Mpr(r_roof_neg['As_provided'], 30.0, r_roof_neg['d'])
Ln_beam = GEOMETRY['L_bay_m'] - 0.40
Ve_hinge_1F_beam = 2*Mpr_1F_beam/Ln_beam
Ve_hinge_roof_beam = 2*Mpr_roof_beam/Ln_beam

# 塑鉸區間距(強度需求 vs 排列上限, 取較小值)
def beam_stirrup_spacing(Ve_hinge, d_beam, db_long=1.9):
    Vn_req = Ve_hinge/0.75
    Vs_req_kgf = Vn_req/9.80665e-3
    s_from_Vs = (2*0.7133*4200.0*d_beam)/Vs_req_kgf
    s_max_seismic = min(d_beam/4, 6*db_long, 15.0)
    return min(s_from_Vs, s_max_seismic)

s_1F_beam = beam_stirrup_spacing(Ve_hinge_1F_beam, r_1F_neg['d'])
s_roof_beam = beam_stirrup_spacing(Ve_hinge_roof_beam, r_roof_neg['d'])
print(f"1F樑: Mpr={Mpr_1F_beam:.2f}, Ve_hinge={Ve_hinge_1F_beam:.2f}kN, 箍筋間距={s_1F_beam:.2f}cm")
print(f"屋頂樑: Mpr={Mpr_roof_beam:.2f}, Ve_hinge={Ve_hinge_roof_beam:.2f}kN, 箍筋間距={s_roof_beam:.2f}cm")


1F樑: Mpr=123.00, Ve_hinge=43.93kN, 箍筋間距=10.95cm
屋頂樑: Mpr=123.00, Ve_hinge=43.93kN, 箍筋間距=10.95cm


In [6]:

# --- 柱端圍束 ---
Ln_1F_col_cm = (GEOMETRY['h1_m'] - 0.50)*100
Ln_2F_col_cm = (GEOMETRY['h2_m'] - 0.50)*100
conf_1F = design_column_confinement(40.0, Ln_1F_col_cm, 1.6)
conf_2F = design_column_confinement(40.0, Ln_2F_col_cm, 1.6)
print(f"1F柱圍束: lo={conf_1F['lo']:.1f}cm, s={conf_1F['s_final']:.2f}cm")
print(f"2F柱圍束: lo={conf_2F['lo']:.1f}cm, s={conf_2F['s_final']:.2f}cm")


1F柱圍束: lo=50.0cm, s=3.96cm
2F柱圍束: lo=50.0cm, s=3.96cm


In [7]:

# --- 梁柱接頭剪力(1F跟屋頂都用同一組公式, 因為梁Mpr相同) ---
def joint_shear_check(As_top, Mpr_beam, h1_or_h2_avg):
    T1_kN = (As_top*5250.0)*9.80665e-3
    Vcol_kN = Mpr_beam/h1_or_h2_avg
    Vj_kN = T1_kN - Vcol_kN
    fc_psi = 280.0*14.223
    Aj_in2 = 1600.0/6.4516
    Vn_kN = (10*(fc_psi**0.5)*Aj_in2/1000)*4.448
    phiVn_kN = 0.85*Vn_kN
    return dict(Vj_kN=Vj_kN, phiVn_kN=phiVn_kN, ratio=phiVn_kN/Vj_kN)

joint_1F = joint_shear_check(r_1F_neg['As_provided'], Mpr_1F_beam, (GEOMETRY['h1_m']+GEOMETRY['h2_m'])/2)
joint_roof = joint_shear_check(r_roof_neg['As_provided'], Mpr_roof_beam, GEOMETRY['h2_m'])
print(f"1F接頭: Vj={joint_1F['Vj_kN']:.2f}, phiVn={joint_1F['phiVn_kN']:.2f}, ratio={joint_1F['ratio']:.2f}")
print(f"屋頂接頭: Vj={joint_roof['Vj_kN']:.2f}, phiVn={joint_roof['phiVn_kN']:.2f}, ratio={joint_roof['ratio']:.2f}")


1F接頭: Vj=259.87, phiVn=591.71, ratio=2.28
屋頂接頭: Vj=259.87, phiVn=591.71, ratio=2.28


## 組裝成 `design_freeze.json`

In [8]:

design_freeze = {
    "status": "FROZEN",
    "source_repo": "taiwan-seismic-code-calc",
    "source_pipeline": "Case-04.7(Stage 2/3, 4-tool verified canonical elastic model) -> Stage 4/5 RC design + seismic detailing",
    "geometry": GEOMETRY,
    "materials": MATERIALS,
    "demands": DEMANDS,
    "members": {
        "1F_column": {
            "section_cm": "40x40", "rho": result_1F_col['rho'],
            "As_total_cm2": round(result_1F_col['As_total'], 2),
            "governing_factor": "ACI 318 minimum reinforcement ratio (not strength demand)",
            "Mn_kNm": round(result_1F_col['Mn'], 2),
            "utilization": round(result_1F_col['r']['utilization'], 4),
            "confinement": {"lo_cm": conf_1F['lo'], "tie_spacing_cm": round(conf_1F['s_final'], 2),
                             "governing_factor": "Ash/s (transverse reinforcement ratio), not shear demand"},
        },
        "2F_column": {
            "section_cm": "40x40", "rho": result_2F_col['rho'],
            "As_total_cm2": round(result_2F_col['As_total'], 2),
            "governing_factor": "ACI 318 minimum reinforcement ratio",
            "Mn_kNm": round(result_2F_col['Mn'], 2),
            "utilization": round(result_2F_col['r']['utilization'], 4),
            "confinement": {"lo_cm": conf_2F['lo'], "tie_spacing_cm": round(conf_2F['s_final'], 2),
                             "governing_factor": "Ash/s (transverse reinforcement ratio)"},
        },
        "1F_beam": {
            "section_cm": "30x50",
            "top_As_cm2": round(r_1F_neg['As_provided'], 2), "top_bar": f"{r_1F_neg['bar_size']}x{r_1F_neg['n_bars']}",
            "bottom_As_cm2": round(r_1F_pos['As_provided'], 2), "bottom_bar": f"{r_1F_pos['bar_size']}x{r_1F_pos['n_bars']}",
            "governing_factor": "ACI 318 minimum reinforcement ratio",
            "Mpr_kNm": round(Mpr_1F_beam, 2),
            "capacity_design_shear_kN": round(Ve_hinge_1F_beam, 2),
            "stirrup_spacing_hinge_zone_cm": round(s_1F_beam, 2),
            "stirrup_governing_factor": "seismic hinge-zone spacing limit (Vc=0 condition met), not shear strength",
        },
        "roof_beam": {
            "section_cm": "30x50",
            "top_As_cm2": round(r_roof_neg['As_provided'], 2), "top_bar": f"{r_roof_neg['bar_size']}x{r_roof_neg['n_bars']}",
            "bottom_As_cm2": round(r_roof_pos['As_provided'], 2), "bottom_bar": f"{r_roof_pos['bar_size']}x{r_roof_pos['n_bars']}",
            "governing_factor": "ACI 318 minimum reinforcement ratio",
            "Mpr_kNm": round(Mpr_roof_beam, 2),
            "capacity_design_shear_kN": round(Ve_hinge_roof_beam, 2),
            "stirrup_spacing_hinge_zone_cm": round(s_roof_beam, 2),
        },
    },
    "scwb": {
        "1F_joint": {"sum_Mnc_kNm": round(sum_Mnc_1F_joint, 2), "ratio": round(scwb_1F_ratio, 2),
                      "status": "PASS" if scwb_1F_ratio >= 1.2 else "FAIL", "code_ref": "ACI 318-19 18.7.3.1"},
        "roof_joint": {"status": "EXEMPT", "reason": "column discontinuous above connection AND Pu < Ag*fc'/10",
                        "code_ref": "ACI 318-19 18.7.3.1 exception (new in 318-19, verified via web search)"},
    },
    "joint_shear": {
        "1F_joint": {"Vj_kN": round(joint_1F['Vj_kN'],2), "phiVn_kN": round(joint_1F['phiVn_kN'],2),
                      "ratio": round(joint_1F['ratio'],2), "status": "PASS", "code_ref": "ACI 318-19 18.8.4"},
        "roof_joint": {"Vj_kN": round(joint_roof['Vj_kN'],2), "phiVn_kN": round(joint_roof['phiVn_kN'],2),
                        "ratio": round(joint_roof['ratio'],2), "status": "PASS", "code_ref": "ACI 318-19 18.8.4"},
    },
    "assumptions": [
        "Stage 2/3 canonical elastic model: 單榀2D substructure, F1/F2側向力直接施加於樓層節點, 樓板內力分配(多榀構架之間)不在這個模型求解範圍內",
        "D+L重力載重集中施加在柱頂節點, 非梁上分佈載重, 導致1F樑/屋頂樑D+L情況下彎矩接近0",
        "有效勁度折減(0.35Ig/0.7Ig)是這個分析目的下宣告的建模假設, 不是唯一或絕對正確的RC真實勁度",
        "柱斷面(40x40cm)、梁斷面(30x50cm)是Case-03.5工程師手動選定的初步試設尺寸, 不是本次Stage 4的尺寸最佳化結果",
        "未涵蓋範圍(不在這次凍結內): 錨定/搭接長度、柱主筋沿全長排筋細節、接頭橫向鋼筋排列細節",
    ],
    "code_references": ["ACI 318-19 Chapter 18 (Special moment frames)", "ACI 318-19 22.4 (PM interaction)",
                          "ACI 318-19 9.6.1.2/10.6.1.1 (minimum reinforcement)"],
}

with open('design_freeze.json', 'w', encoding='utf-8') as f:
    json.dump(design_freeze, f, indent=2, ensure_ascii=False)

print("design_freeze.json 產生完成")
print(json.dumps(design_freeze, indent=2, ensure_ascii=False)[:800])


design_freeze.json 產生完成
{
  "status": "FROZEN",
  "source_repo": "taiwan-seismic-code-calc",
  "source_pipeline": "Case-04.7(Stage 2/3, 4-tool verified canonical elastic model) -> Stage 4/5 RC design + seismic detailing",
  "geometry": {
    "L_bay_m": 6.0,
    "h1_m": 3.5,
    "h2_m": 3.5,
    "col_section_cm": "40x40",
    "beam_section_cm": "30x50"
  },
  "materials": {
    "fc_kgf_cm2": 280.0,
    "fy_kgf_cm2": 4200.0,
    "E_kPa": 24630000.0,
    "stiffness_reduction": "beam 0.35Ig, column 0.7Ig (ACI 318 declared assumption, not unique true value)"
  },
  "demands": {
    "1F_column": {
      "Pu_kN": 308.85,
      "Mu_kNm": 32.13,
      "Vu_kN": 12.94,
      "governing_combo": "D+L+E"
    },
    "2F_column": {
      "Pu_kN": 153.19,
      "Mu_kNm": 11.06,
      "Vu_kN": 7.95,
      "governing_combo": "D+L+E


## 驗證:`JSON` 內容跟原始 `Stage 4/5` `notebook` 的數字一致

不是產生一份看起來合理但沒對過的 `JSON`,逐項比對關鍵數字。

In [9]:

checks = [
    (result_1F_col['Mn'], 159.97, "1F柱Mn"),
    (result_2F_col['Mn'], 136.83, "2F柱Mn"),
    (sum_Mnc_1F_joint, 296.80, "1F接頭sum(Mnc)"),
    (Ve_hinge_1F_beam, 43.93, "1F樑capacity-design shear"),
    (joint_1F['Vj_kN'], 259.87, "1F接頭Vj"),
    (conf_1F['s_final'], 4.0, "1F柱圍束間距"),
]
all_pass = True
for computed, reference, label in checks:
    diff = abs(computed-reference)
    ok = diff < 0.5
    all_pass = all_pass and ok
    print(f"{label}: 重新計算={computed:.2f}, 原notebook={reference}, 差異={diff:.3f} {'OK' if ok else 'MISMATCH'}")

assert all_pass, "design_freeze.json的數字應該跟原始Stage 4/5 notebook一致"
print("\n[PASS] design_freeze.json的所有關鍵數字都跟原始Stage 4/5 notebook一致")


1F柱Mn: 重新計算=159.97, 原notebook=159.97, 差異=0.003 OK
2F柱Mn: 重新計算=136.83, 原notebook=136.83, 差異=0.005 OK
1F接頭sum(Mnc): 重新計算=296.80, 原notebook=296.8, 差異=0.002 OK
1F樑capacity-design shear: 重新計算=43.93, 原notebook=43.93, 差異=0.002 OK
1F接頭Vj: 重新計算=259.87, 原notebook=259.87, 差異=0.004 OK
1F柱圍束間距: 重新計算=3.96, 原notebook=4.0, 差異=0.037 OK

[PASS] design_freeze.json的所有關鍵數字都跟原始Stage 4/5 notebook一致


## 小結

- `design_freeze.json` 由重新執行的計算組裝而成,不是手動謄寫,
  逐項比對過跟原始 `Stage 4/5 notebook` 的數字一致
- 涵蓋範圍誠實列在 `JSON` 的 `assumptions` 欄位裡,沒有涵蓋的部分
  (錨定搭接、全長排筋細節)也明確列出,不是隱藏
- 這是後續非線性分析(`Case-06.7`/`06.8`/`06.9`)該讀取的單一資料
  來源——不再各自假設 $\rho=0.02$,直接讀這份凍結後的真實配筋